# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilip-Git18/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My Lane as an ML Task

### Selected Lane
Predict content that is likely to decline in search performance.

### ML Task Type
This project is a **binary classification** problem because each content page belongs to one of two categories:

- Declining
- Not Declining

The goal is to use historical search and engagement metrics to predict whether a content page is likely to decline. This helps SEO teams identify pages that may need attention before significant traffic loss occurs.

## Target or Proxy

The target variable for this project is **is_declining_label**.

This label represents whether a content page is declining in search performance. It is an **observed label** provided in the dataset rather than a manually created rule.

Possible values:

- 1 = Declining
- 0 = Not Declining

The model will learn relationships between historical content performance metrics and this observed outcome.

## Success Metric

The primary evaluation metric is **F1-score** because both false positives and false negatives have business costs.

Additional evaluation metrics include:

- Accuracy
- Precision
- Recall
- ROC-AUC

A good model should correctly identify declining pages while minimizing unnecessary recommendations for stable pages.

In [10]:
!git clone https://github.com/Dilip-Git18/flyrank-ml-internship.git

import os
os.chdir("/content/flyrank-ml-internship")

print(os.getcwd())

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 131 (delta 45), reused 105 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 1.85 MiB | 12.30 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/flyrank-ml-internship


In [11]:
import os

for root, dirs, files in os.walk("/content/flyrank-ml-internship"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [12]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


In [13]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Create binary target
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows")
display(df.head())

print("\nTarget Variable Distribution")
print(df["is_declining_label"].value_counts())

print("\nTrend Direction Distribution")
print(df["trend_direction"].value_counts())

print("\nAverage CTR:", round(df["ctr"].mean(), 2))
print("Average Search Position:", round(df["avg_position"].replace(0, pd.NA).dropna().mean(), 2))
print("Average Engagement Rate:", round(df["engagement_rate"].mean(), 2))

print("\nNumber of Rows:", len(df))
print("Number of Columns:", len(df.columns))

print("\nUnit of Analysis")
print("Each row represents one content page.")

print("\nExample Features:")
features = [
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_age_days"
]
print(features)

print("\nTarget Column:")
print("is_declining_label (created from trend_direction)")

Dataset Shape: (30000, 45)

First 5 Rows


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1



Target Variable Distribution
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Trend Direction Distribution
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Average CTR: 0.51
Average Search Position: 17.03
Average Engagement Rate: 2.53

Number of Rows: 30000
Number of Columns: 45

Unit of Analysis
Each row represents one content page.

Example Features:
['search_volume', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'content_age_days']

Target Column:
is_declining_label (created from trend_direction)


## The Unit of Analysis

The unit of analysis is **one content page**.

Each row in the dataset represents a single content page and includes search performance, engagement metrics, content characteristics, and historical trend information.

The machine learning model predicts whether that individual content page is likely to decline in search performance.

## Why ML Beats a Fixed Rule

A simple rule such as "update every page with CTR below 1%" would ignore many important factors, including search position, engagement rate, search volume, content age, and content characteristics.

Machine learning can evaluate many variables together and identify complex patterns that are difficult to capture using fixed rules.

The model is intended to provide decision support by helping SEO teams prioritize pages for review rather than replacing human judgment.

In [14]:
print("Notebook completed successfully.")
print("ML Task Type: Binary Classification")
print("Target Variable: is_declining_label")
print("Unit of Analysis: One content page")

Notebook completed successfully.
ML Task Type: Binary Classification
Target Variable: is_declining_label
Unit of Analysis: One content page


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.